# Non-stationary SWTP and SVSL from LQI

This notebook implements the analytical framework used to evaluate the **societal willingness-to-pay (SWTP)** and the **societal value of a statistical life (SVSL)** derived from the **Life Quality Index (LQI)** under **non-stationary socioeconomic and demographic conditions**.

The implementation accompanies the manuscript:

**Cao (2026), *Non-stationary risk acceptance in light of climate change*. In: 4th International Conference on Natural Hazards & Infrastructure (ICONHIC2026).**

The objective of this notebook is to reproduce the numerical results presented in the paper and to demonstrate how **economic growth, demographic change, and mortality improvements** influence risk-acceptance metrics over time.



## Background

In the LQI framework, acceptable safety levels can be derived using the principle of **marginal life-saving costs**. The resulting **societal willingness-to-pay (SWTP)** represents the marginal societal expenditure that preserves welfare when mortality risk is reduced. Closely related is the **societal value of a statistical life (SVSL)**, which quantifies the value associated with a marginal reduction in mortality risk.

Most applications of the LQI assume **time-stationary socioeconomic parameters**, such as constant life expectancy, age distributions, and economic output. However, infrastructure systems are designed to operate over long time horizons during which **economic conditions, demographics, and mortality patterns evolve**.

This notebook implements a **time-dependent formulation of SWTP and SVSL**, allowing these quantities to be evaluated under evolving socioeconomic conditions.



## Scope of the analysis

The notebook performs the following steps:

1. Compute **discounted remaining life expectancy** under stationary and non-stationary mortality assumptions.
2. Evaluate the **demographic parameters** used in the LQI-based SWTP formulation.
3. Compute **SWTP and SVSL at the decision time**.
4. Compute **time-averaged SWTP and SVSL over a decision horizon**.
5. Compare **stationary and non-stationary valuations** across countries.

The analysis is performed for selected countries (Brazil, China, India, South Africa, Switzerland, and the United States of America) using projections of **population, mortality, and economic output**.



## Data sources

The socioeconomic and demographic inputs used in the analysis are derived from publicly available datasets:

- **Wittgenstein Centre / IIASA population projections**
- **Shared Socioeconomic Pathways (SSP) scenarios**
- **World Bank labour participation data**
- **Penn World Table (PWT 9.1)**

These datasets provide the required projections of:

- population age distributions
- mortality rates
- GDP per capita
- labour participation and working hours.



## Modelling assumptions

The implementation follows the modelling assumptions described in the manuscript:

- constant mortality reduction across ages,
- piecewise-constant hazard rates within 5-year age bins,
- trapezoidal numerical integration,
- constant work–leisure ratio,
- socioeconomic projections based on **SSP2**.



## Structure of the notebook

The notebook is organised as follows:

1. **Methodology**  
   Implementation of the LQI-based formulation for SWTP and SVSL.

2. **Data loading and preprocessing**  
   Import of demographic and socioeconomic data.

3. **Computation of demographic parameters**

4. **Computation of SWTP and SVSL**

5. **Time-averaged valuation over the decision horizon**

6. **Results and comparison with stationary formulations**


The notebook is intended as a **reproducible implementation** of the analytical framework used in the manuscript and can be adapted to evaluate alternative socioeconomic scenarios or countries.


---

# Methodology

This notebook computes the **societal willingness to pay for mortality risk reductions** using the **Life Quality Index (LQI) framework** under stationary and non-stationary socioeconomic conditions. The methodology combines demographic data, mortality projections, GDP projections, and time discounting to compute the following quantities:

* discounted remaining life expectancy $e_{\mathrm{d}}$,
* the marginal value of mortality risk reduction $C_\Delta$,
* the value of a statistical life SVSL,
* the societal willingness to pay SWTP,
* time-averaged values over a decision horizon.

The formulation follows the standard LQI approach but allows for **non-stationary mortality, demography, and economic conditions**.



## Discounting

Economic growth determines the social time-discounting rate. The discount rate is defined as

$
\gamma_M(t) = \varepsilon \delta(t) + \rho
$

where

* $\varepsilon$ is the elasticity of marginal utility of consumption,
* $\delta(t)$ is the GDP growth rate,
* $\rho$ is the pure rate of time preference.

GDP growth is obtained from projected GDP per capita (g(t)):

$
\delta(t) = \frac{g(t+1)}{g(t)} - 1
$

The corresponding discount factor is

$
D(t) = \exp\left(-\int_0^t \gamma_M(u)\mathrm{d}u\right)
$



## Mortality and survival

Mortality is represented through the hazard rate

$
\mu(a,t)
$

which depends on age $a$ and calendar time $t$. The survival function for a cohort starting at age $a_0$ at time $t_0$ is

$
S(s \mid a_0,t_0) =
\exp\left(-\int_0^s \mu(a_0+u, t_0+u)\mathrm{d}u\right)
$

where $s$ is the time elapsed since the decision year.

Mortality data are provided in discrete age and time bins. The hazard rate within each bin is obtained from the survival probability $p$ over an interval of length $L$:

$
\mu = -\frac{\ln p}{L}
$



## Discounted remaining life expectancy

The **discounted remaining life expectancy** is defined as

$
e_{\mathrm{d}}(a_0,t_0) =
\int_0^T D(s) S(s \mid a_0,t_0) \mathrm{d}s
$

where

* $T$ is the evaluation horizon,
* $D(s)$ is the discount factor,
* $S(s)$ is the survival probability.



## Sensitivity to mortality risk reduction

Consider a small reduction in mortality hazard $\Delta$. The derivative of the discounted remaining life expectancy with respect to this reduction is

$
\frac{\partial e_{\mathrm{d}}}{\partial \Delta}\bigg|_{\Delta=0} \int_0^T s D(s) S(s \mid a_0,t_0) \mathrm{d}s
$

This quantity measures the marginal life-years gained from a small reduction in mortality.



## Population age distribution

The age distribution of the population is represented by a density

$
h(a,t)
$

derived from population projections. In discrete form, the population is represented by age bins $A_i$ with probability mass

$
h(a,t)\doteq f_A(a_i,t)
$

such that

$
\sum_i f_A(a_i,t) = 1.
$

Within each age bin, the contributions of males and females are combined using the corresponding population shares.



## Age-averaged quantities

The discounted remaining life expectancy averaged over the population age distribution is

$
E_A[e_{\mathrm{d}}] =
\sum_i f_A(a_i,t_0) , e_{\mathrm{d}}(a_i,t_0).
$

Similarly, the population-averaged marginal benefit of mortality reduction is

$
C_\Delta =
\sum_i f_A(a_i,t_0)
\frac{\partial e_{\mathrm{d}} / \partial \mathrm{d}m}{e_{\mathrm{d}}}.
$



## Value of a statistical life

Within the LQI framework, the **value of a statistical life** is

$
\text{SVSL}(t_0) =
\frac{g(t_0)}{q(t_0)} E_A[e_{\mathrm{d}}]
$

where

* $g(t_0)$ is GDP per capita,
* $q(t_0)$ is the demographic constant reflecting the trade-off between work and leisure.



## Societal willingness to pay

The societal willingness to pay for a unit mortality reduction is

$
\text{SWTP}(t_0) =
\frac{g(t_0)}{q(t_0)} C_\Delta \mathrm{d}m.
$

This quantity represents the maximum fraction of economic output that society would rationally allocate to mortality risk reduction.



## Time-averaged valuation over a decision horizon

Infrastructure decisions often affect multiple future generations. The methodology therefore evaluates the valuation quantities along a **remaining horizon formulation**

$
(t_0+s ; T-s)
$

for $s \in [0,T]$.

At each time $s$:

* the evaluation year becomes $t_0+s$,
* the remaining horizon becomes $T-s$.

The time-averaged values are

$
\text{SVSL}_T =
\frac{1}{T}\int_0^T \frac{g(t_0+s)}{q(t_0+s)} E_A[e_{\mathrm{d}}(t_0+s;T-s)] \mathrm{d}s
$

$
\text{SWTP}_T =
\frac{1}{T}\int_0^T \frac{g(t_0+s)}{q(t_0+s)} C_\Delta(t_0+s;T-s) \mathrm{d}s \mathrm{d}m
$

which represent the average valuation across successive future societies over the decision horizon.



## Stationary vs non-stationary formulation

Two variants are evaluated:

### Stationary formulation

Mortality and demographic conditions are assumed constant at the decision year $t_0$:

$
\mu(a,t) = \mu(a,t_0)
$

### Non-stationary formulation

Mortality, demography, and economic conditions evolve according to projections:

$
\mu(a,t), \quad h(a,t), \quad g(t)
$

This formulation accounts for changing population structure, longevity, and economic growth.





---

# Numerical implementation and discretization

The analytical expressions introduced above involve integrals over time. These integrals are evaluated numerically using a discrete time grid and trapezoidal quadrature.



## Time discretization

The continuous time horizon

$
s \in [0,T]
$

is discretized using a uniform time step

$
\Delta t
$

which defines the grid

$
s_i = i,\Delta t , \qquad i = 0,1,\dots,n
$

with

$
n = \left\lfloor \frac{T}{\Delta t} \right\rfloor .
$

The numerical integration step $\Delta t$ controls the resolution of the cohort evolution and the numerical accuracy of the integrals.



## Trapezoidal quadrature

All time integrals are evaluated using the trapezoidal rule. For a generic integral

$
I = \int_0^T f(s)ds
$

the numerical approximation becomes

$
I \approx
\sum_{i=0}^{n-1}
\frac{f(s_i) + f(s_{i+1})}{2}\Delta t .
$

This scheme provides second-order accuracy.



## Survival integration

The survival probability is expressed in terms of the cumulative hazard

$
H(s) = \int_0^s \mu(a_0+u, t_0+u)\mathrm{d}u
$

with

$
S(s) = \exp(-H(s)).
$

The cumulative hazard is computed numerically using trapezoidal integration of the hazard rate:

$
H_{i+1} =
H_i +
\frac{\mu_i + \mu_{i+1}}{2}\Delta t .
$

This produces the survival curve

$
S_i = \exp(-H_i).
$



## Discount factor integration

The discount factor is defined as

$
D(s) =
\exp\left(-\int_0^s \gamma_M(u)\mathrm{d}u\right).
$

The cumulative discount exponent is evaluated using the trapezoidal rule:

$
G_{i+1} =
G_i +
\frac{\gamma_i + \gamma_{i+1}}{2}\Delta t .
$

The discount factor at each grid point is therefore

$
D_i = \exp(-G_i).
$



## Evaluation of discounted life expectancy

Using the discrete time grid, the discounted remaining life expectancy becomes

$
e_d \approx
\sum_{i=0}^{n-1}
\frac{D_i S_i + D_{i+1} S_{i+1}}{2}\Delta t .
$

Similarly, the derivative with respect to mortality reduction becomes

$
\frac{\partial e_d}{\partial \Delta}\Big|\cdot0
\approx
\sum\cdot{i=0}^{n-1}
\frac{s_i D_i S_i + s_{i+1} D_{i+1} S_{i+1}}{2}\Delta t .
$



## Handling of discretized demographic and mortality data

The input datasets for mortality and demography are available only at discrete resolutions.

## Mortality projections

Mortality data are provided as survival probabilities for

* 5-year age bins
* 5-year time periods.

For each bin the hazard rate is assumed constant:

$
\mu = -\frac{\ln p}{L}
$

where

* $p$ is the survival probability over the interval,
* $L$ is the bin width.

When the cohort evolves, the hazard rate is updated whenever the age or period crosses a bin boundary.



### Population age distribution

Population projections are available at discrete years (typically 5-year intervals).
For intermediate years the demographic structure is assumed piecewise constant:

$
h(a,t) = h(a,t_k)
\quad
t_k \le t < t_{k+1}
$

where $t_k$ are the years available in the demographic dataset.



### GDP projections

GDP projections are typically available at multi-year intervals. To obtain annual discount rates, the GDP series is interpolated in logarithmic space:

$
\ln g(t)
$

is linearly interpolated between data points, which corresponds to assuming constant annual growth within each interval.

The annual growth rate used in the discounting formula is therefore

$
\delta(t) =
\frac{g(t+1)}{g(t)} - 1 .
$



## Choice of integration step

Although the input datasets are provided at multi-year resolution, a smaller integration step improves numerical accuracy because

* hazard rates change when age crosses bin boundaries,
* discount rates vary annually,
* survival and discounting interact exponentially.

In practice, a step size of

$
\Delta t = 1 \text{ year}
$

provides stable numerical results, while larger steps can introduce discretization errors.

---



# Dataset (`input.db`)

The SQLite database `input.db` contains the socioeconomic, demographic, and labour parameters used to compute the **societal willingness-to-pay (SWTP)** in the Life Quality Index (LQI) framework.

The dataset combines projections from the **Shared Socioeconomic Pathways (SSPs)** with baseline labour statistics used to estimate the labour–leisure trade-off.



## SSP economic and demographic projections

**Source**

IIASA SSP database (DOI: 10.5281/zenodo.7767425)  
https://ssp.apps.ece.iiasa.ac.at/documentation/basic-drivers


The SSP dataset provides projections of:

- **GDP per capita** (`g_SSPs`)  
- **Life tables and mortality rates** (`female_l`, `male_l`)

These variables describe future economic development and demographic change and enter the SWTP and SVSL.



## Labour parameters

Additional baseline parameters are used to estimate the labour–leisure trade-off in the LQI model.

**Labour participation rate**

World Bank – World Development Indicators  
https://data.worldbank.org

Table: `P15+_all`

**Working hours per worker**

Penn World Table 9.1  
https://www.rug.nl/ggdc/productivity/pwt/

Table: `H`

These parameters are assumed constant and are used to compute the labour–leisure ratio and the parameter \(q\) in the LQI framework.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import lqi_swtp as lqi


In [ ]:
# ----------------------------
# 1) Configuration
# ----------------------------
cfg = lqi.Config(
    db_path=Path("input_Brazil_China_India_South_Africa_Switzerland_United_States_of_America.db"),
    country="Brazil",           # "Brazil", "China", "India", "South Africa", "Switzerland", "United States of America"
    ssp="SSP2",                 # "SSP1", "SSP2", "SSP3", "SSP4", "SSP5"
    t0=2050,                    # reference year for SVSL and SWTP calculations
    t0_age=2050,                # reference age for SVSL and SWTP calculations 
    T=50.0,                     # decision horizon in years for time-averaged SVSL_T and SWTP_T calculations
    dt=5,                       # time step in years for time-averaged SVSL_T and SWTP_T calculations
    epsilon=2.0,                # elasticity of marginal utility of consumption 
    rho=0.02,                   # pure rate of time preference
    gamma_M_const=0.02,         # used only as fallback if GDP growth is unavailable
    beta_year=2017,             # reference year for beta parameter in q(t0) calculation (should be <= t0
)

# ----------------------------
# 2) Load tables from input.db
# ----------------------------
tables = lqi.load_required_tables(cfg.db_path)

surv_f = tables["Survival_female"]
surv_m = tables["Survival_male"]
l_f = tables["female_l"]
l_m = tables["male_l"]
g_ssp = tables["g_SSPs"]
beta_tbl = tables["beta"]
P15_tbl = tables["P15+_all"]
H_tbl = tables["H"]
pop_age_db = tables["population_age"]

# ----------------------------
# 3) Age & sex weights
# ----------------------------
pop_df, age_bins = lqi.build_age_sex_weights(pop_age_db, cfg)

# ----------------------------
# 4) GDP-driven discounting (gamma_M(t) = epsilon*delta(t) + rho)
# ----------------------------
gamma_by_year, discount_factors = lqi.build_discount_factors(cfg, g_ssp)

# ----------------------------
# 5) Survival tables -> hazard lookup
# ----------------------------
lut_f, periods, ages = lqi.build_surv_lookup(surv_f, cfg)
lut_m, _, _ = lqi.build_surv_lookup(surv_m, cfg)

mu_kwargs = dict(lut_f=lut_f, lut_m=lut_m, periods=periods, ages=ages)

# ----------------------------
# 6) Age-averaged e_d and C_Delta (stationary vs non-stationary)
# ----------------------------
detail_ns, E_ed_ns, C_Delta_ns = lqi.compute_age_averages(
    cfg, age_bins, stationary=False, mu_kwargs=mu_kwargs, discount_factors=discount_factors
)
detail_st, E_ed_st, C_Delta_st = lqi.compute_age_averages(
    cfg, age_bins, stationary=True, mu_kwargs=mu_kwargs, discount_factors=discount_factors
)

# ----------------------------
# 7) SVSL and SWTP at t_0
# ----------------------------
g_t0 = lqi.compute_g_t0(cfg, g_ssp)

pop_age_db = tables["population_age"]

q_t0, beta_val, w_frac = lqi.compute_q_t0(
    cfg,
    pop_age=pop_age_db,
    l_f=l_f,
    l_m=l_m,
    P15_tbl=P15_tbl,
    H_tbl=H_tbl,
    beta_tbl=beta_tbl,
    periods=periods,
)

vals = lqi.compute_svsl_swtp(
    cfg,
    g_t0=g_t0,
    q_t0=q_t0,
    E_ed_ns=E_ed_ns,
    E_ed_st=E_ed_st,
    C_Delta_ns=C_Delta_ns,
    C_Delta_st=C_Delta_st,
)

# ----------------------------
# 8) Time-averaged SVSL_T and SWTP_T over the decision horizon (Eqs. 14–15)
# ----------------------------

ta_ns = lqi.time_average_svsl_swtp(cfg, tables=tables, g_ssp=g_ssp, use_stationary_mortality=False, nonstationary_demography=True)
ta_st = lqi.time_average_svsl_swtp(cfg, tables=tables, g_ssp=g_ssp, use_stationary_mortality=True, nonstationary_demography=False)

# ----------------------------
# 9) Summary table
# ----------------------------
summary = pd.DataFrame([{
    "Country": cfg.country,
    "Scenario": cfg.ssp,
    "t0": cfg.t0,
    "t0_age": cfg.t0_age,
    "T (yr)": cfg.T,
    "dt (yr)": cfg.dt,
    "epsilon": cfg.epsilon,
    "rho": cfg.rho,
    "g(t0) GDP|PPP pc": g_t0,
    "q(t0)": q_t0,
    "E_A[e_{\mathrm{d}}] non-stationary (yr)": E_ed_ns,
    "E_A[e_{\mathrm{d}}] stationary (yr)": E_ed_st,
    "C_Δ non-stationary (yr^2)": C_Delta_ns,
    "C_Δ stationary (yr^2)": C_Delta_st,
    "SVSL non-stationary": vals["SVSL_ns"],
    "SVSL stationary": vals["SVSL_st"],
    "SWTP unit (Δ=1) non-stationary": vals["SWTP_unit_ns"],
    "SWTP unit (Δ=1) stationary": vals["SWTP_unit_st"],
    "SVSL_T non-stationary": ta_ns["SVSL_T"],
    "SWTP_T unit (Δ=1) non-stationary": ta_ns["SWTP_T"],
}])

summary.T

## Plots

### Age distribution density $h(a,t_0)$

In [ ]:

import cmcrameri.cm as cm
import matplotlib.pyplot as plt
try:
    plt.rcParams["text.usetex"] = True
except Exception:
    plt.rcParams["text.usetex"] = False
plt.rcParams["font.size"] = 16
plt.rc("font", family="times")

fig, ax = plt.subplots(figsize=tuple(plt.figaspect(0.8) * 1.25), dpi=80)

years = [2025, 2050, 2075]
ssp = 'SSP2'
country = 'United States of America'
for year in years:
    cfg = lqi.Config(
        db_path=Path("input.db"),
        country=country,
        ssp=ssp,
        t0=2025,
        t0_age=year,
        T=50.0,
        dt=1.0,
        epsilon=2.0,
        rho=0.02,
        gamma_M_const=0.02,  # used only as fallback if GDP growth is unavailable
        beta_year=2017,
    )
    _, age_bins = lqi.build_age_sex_weights(pop_age_db, cfg)

    # Age weights (population mass) plot
    ax.bar(age_bins["a_mid"], age_bins["fA_mass"], width=age_bins["L"], 
           align="center", 
           color=cm.batlow(years.index(year) / len(years)),
           edgecolor=(0.3,0.3,0.3),
           alpha=0.5,
           label=r'$t_0$ = ' + str(year))
ax.set_xlabel(r"$a$ (years)")
ax.set_ylabel(r"$h(a,t_0)$")
ax.legend()
fig.tight_layout()
titlePDF = 'figures/fig_' + 'popdensity' + ssp + '_' + country + '_' + '.pdf'
plt.savefig(titlePDF, bbox_inches='tight', dpi=300)
titlePDF = 'figures/fig_' + 'popdensity' + ssp + '_' + country + '_' + '.svg'
plt.savefig(titlePDF, bbox_inches='tight', dpi=300)
plt.show()

### Time-discounted remaining life expectancy $e_{\mathrm{d}}$

In [ ]:

fig, ax = plt.subplots(figsize=tuple(plt.figaspect(0.8) * 1.25), dpi=80)
country='United States of America'
years = [2025, 2050, 2075]
for year in years:
    cfg = lqi.Config(
        db_path=Path("input.db"),
        country=country,
        ssp="SSP2",
        t0=year,
        t0_age=year,
        T=50.0,
        dt=1.0,
        epsilon=2.0,
        rho=0.02,
        gamma_M_const=0.02,  # used only as fallback if GDP growth is unavailable
        beta_year=2017,
    )

    tables = lqi.load_required_tables(cfg.db_path)
    pop_age_db = tables["population_age"]
    _, age_bins = lqi.build_age_sex_weights(pop_age_db, cfg)

    detail_ns, E_ed_ns, C_Delta_ns = lqi.compute_age_averages(
        cfg, age_bins, stationary=False, mu_kwargs=mu_kwargs, discount_factors=discount_factors
    )
    detail_st, E_ed_st, C_Delta_st = lqi.compute_age_averages(
        cfg, age_bins, stationary=True, mu_kwargs=mu_kwargs, discount_factors=discount_factors
    )
    ax.plot(detail_ns["a_start"], detail_ns["e_{\mathrm{d}}"],
            color=cm.batlow(years.index(year) / len(years)),
            linestyle="-", 
            label=r"$t_0$ = " + str(year) + " non-stationary")
    ax.scatter(detail_ns["a_start"], detail_ns["e_{\mathrm{d}}"],
            color=cm.batlow(years.index(year) / len(years)),
            s=30, alpha=0.5, edgecolors="k")
    ax.plot(detail_st["a_start"], detail_st["e_{\mathrm{d}}"], 
            color=cm.batlow(years.index(year) / len(years)),
            linestyle="--",
            label=r"$t_0$ = " + str(year) + " stationary")
    ax.scatter(detail_st["a_start"], detail_st["e_{\mathrm{d}}"],
            color=cm.batlow(years.index(year) / len(years)),
            s=30, alpha=0.5, edgecolors="k")
ax.set_xlabel(r"$a_0$ (years)")
ax.set_ylabel(r"$e_{\mathrm{d}}$ (years)")
ax.set_xlim([min(detail_ns["a_start"]), max(detail_ns["a_start"])])
ax.set_ylim([0,max(detail_ns["e_{\mathrm{d}}"])])
ax.set_xticks(np.arange(0, 101, 10))
ax.legend()
fig.tight_layout()
titlePDF = 'figures/fig_' + 'ed' + ssp + '_' + country + '_' + '.pdf'
plt.savefig(titlePDF, bbox_inches='tight', dpi=300)
titlePDF = 'figures/fig_' + 'ed' + ssp + '_' + country + '_' + '.svg'
plt.savefig(titlePDF, bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
detail_ns.head()